# GCE Eligibility - 03: MLP Evaluation

Loads `best_model.pth` trained locally on RTX 3060 Ti (CUDA 12.1). Runs test set evaluation and threshold sweep. Logs all results to MLflow.

In [ ]:
import torch
import numpy as np
import mlflow
import mlflow.pytorch
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import torch.nn as nn

VOL    = "/Volumes/ml_final_workspace/default/ml_final"
SPLITS = f"{VOL}/splits"

print(f"PyTorch version: {torch.__version__}")
# cluster has no GPU — model trained on RTX 3060 Ti locally, weights loaded CPU-only here
device = torch.device("cpu")
print(f"Device: {device}")


In [ ]:
# model definition must match training exactly — 14 inputs, 128/64/32 hidden, no output activation
class GCE_MLP(nn.Module):
    def __init__(self, input_dim):
        super(GCE_MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

model = GCE_MLP(input_dim=14)
# map_location='cpu' because cluster has no GPU
model.load_state_dict(torch.load(f"{VOL}/best_model.pth",
                                  map_location="cpu", weights_only=True))
model.eval()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# load test split from 01_data_prep
X_test = np.load(f"{SPLITS}/X_test.npy")
y_test = np.load(f"{SPLITS}/y_test.npy")

X_test_t = torch.tensor(X_test, dtype=torch.float32)
print(f"Test set: {X_test_t.shape}, positives: {int(y_test.sum()):,}")


In [ ]:
# threshold sweep — logs each threshold as a separate MLflow run
# open Experiments tab after this cell to see all 8 runs compared in the UI
mlflow.set_experiment("/gce_eligibility")

THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8]

with torch.no_grad():
    logits = model(X_test_t)
    probs  = torch.sigmoid(logits).numpy()

print(f"{'Threshold':<12} {'F1':<10} {'Precision':<12} {'Recall':<10}")
print("-" * 44)

for thresh in THRESHOLDS:
    binary = (probs >= thresh).astype(int)
    f1   = f1_score(y_test, binary, zero_division=0)
    prec = precision_score(y_test, binary, zero_division=0)
    rec  = recall_score(y_test, binary, zero_division=0)
    acc  = accuracy_score(y_test, binary)
    print(f"{thresh:<12} {f1:<10.4f} {prec:<12.4f} {rec:<10.4f}")

    with mlflow.start_run(run_name=f"mlp_threshold_{thresh}"):
        mlflow.log_params({
            "model": "GCE_MLP",
            "architecture": "14-128-64-32-1",
            "dropout": 0.3,
            "pos_weight": 19.55,
            "threshold": thresh,
            "trained_on": "RTX 3060 Ti / CUDA 12.1",
            "epochs": 24,
            "early_stopping_patience": 5
        })
        mlflow.log_metrics({"f1": f1, "precision": prec, "recall": rec, "accuracy": acc})


In [ ]:
# final test evaluation at selected threshold (0.8)
THRESHOLD = 0.8
test_binary    = (probs >= THRESHOLD).astype(int)
test_f1        = f1_score(y_test, test_binary, zero_division=0)
test_precision = precision_score(y_test, test_binary, zero_division=0)
test_recall    = recall_score(y_test, test_binary, zero_division=0)
test_accuracy  = accuracy_score(y_test, test_binary)

print("=" * 60)
print(f"MLP FINAL TEST RESULTS — Threshold {THRESHOLD}")
print("=" * 60)
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"F1:        {test_f1:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print("=" * 60)


In [ ]:
# log the final selected run as the primary MLP entry
with mlflow.start_run(run_name="mlp_final_t0.8"):
    mlflow.log_params({
        "model": "GCE_MLP",
        "architecture": "14-128-64-32-1",
        "dropout": 0.3,
        "pos_weight": 19.55,
        "threshold": 0.8,
        "optimizer": "Adam",
        "lr": 1e-3,
        "batch_size": 2048,
        "epochs_trained": 24,
        "early_stopping_patience": 5,
        "lr_scheduler": "ReduceLROnPlateau",
        "trained_on": "RTX 3060 Ti / CUDA 12.1"
    })
    mlflow.log_metrics({
        "accuracy":  test_accuracy,
        "f1":        test_f1,
        "precision": test_precision,
        "recall":    test_recall
    })
    mlflow.pytorch.log_model(model, "gce_mlp")
    print("Logged to MLflow. Open Experiments tab to compare all runs.")
